# Практическая работа. Множественная линейная регрессия на телеметрии спутника

**Курс «Машинное обучение в задачах космической отрасли»**  
**Задача:** сравнить простую модель по току `current` с множественной моделью по `current`, `voltage`, `mode` для предсказания температуры `temperature`.

**Что научимся делать:** загружать CSV, кодировать категориальный признак, обучать две регрессии на **одинаковом разделении Train/Test**, сравнивать MAE, MSE, RMSE, R², изучать коэффициенты и получать прогноз для новых значений телеметрии.

> Данные учебные, синтетические. Модель оценивает температуру по параметрам одного наблюдения, а не прогнозирует её через несколько минут.

## 1. Импорт библиотек

Запускайте ячейки последовательно сверху вниз. Достаточно стандартного окружения Google Colab или Jupyter с `pandas`, `numpy`, `matplotlib` и `scikit-learn`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pd.set_option("display.max_columns", None)
plt.rcParams["figure.figsize"] = (9, 5)

## 2. Загрузка CSV

Notebook ищет файл `satellite_linear_regression_600.csv` рядом с собой. В Google Colab, если файла нет, предложит загрузить его с компьютера. **CSV необходимо скачать отдельно вместе с ноутбуком.**

In [ ]:
from pathlib import Path

filename = "satellite_linear_regression_600.csv"
if not Path(filename).exists():
    try:
        from google.colab import files
        print("Выберите файл:", filename)
        files.upload()
    except ImportError:
        raise FileNotFoundError(
            f"Поместите {filename} в папку с notebook и запустите ячейку снова."
        )

if not Path(filename).exists():
    raise FileNotFoundError(f"Файл {filename} не найден после загрузки.")

df = pd.read_csv(filename)
display(df.head())
print("Размер датасета:", df.shape)
print("Пропуски по столбцам:\n", df.isna().sum().to_string())
print("Режимы:", df["mode"].value_counts().to_dict())

## 3. Признаки и целевая переменная

- **Простая модель:** только `current` (ток, А).
- **Множественная модель:** `current`, `voltage` (напряжение, В), `mode` (режим работы).
- **Целевая переменная:** `temperature` (°C).

`mode` — текстовый признак. Линейной регрессии нужны числа, поэтому используем **one-hot encoding**: создадим бинарные столбцы для режимов, один из них оставим базовым. `power` пока не добавляем, поскольку она вычисляется как `current × voltage` и может создавать избыточность.

In [ ]:
required = {"current", "voltage", "mode", "temperature"}
assert required.issubset(df.columns), f"Отсутствуют столбцы: {required - set(df.columns)}"
assert not df[list(required)].isna().any().any(), "В нужных столбцах есть пропуски"

X_simple = df[["current"]].copy()

# One-hot кодирование режима; drop_first=True убирает одну базовую категорию.
X_multiple = pd.get_dummies(
    df[["current", "voltage", "mode"]],
    columns=["mode"],
    drop_first=True,
    dtype=int
)
y = df["temperature"].copy()

print("Простые признаки:", X_simple.columns.tolist())
print("Признаки множественной модели:", X_multiple.columns.tolist())
display(X_multiple.head())

## 4. Одно и то же разделение Train/Test для двух моделей

Используем **одни и те же индексы наблюдений**: 80% Train, 20% Test. Это ключевое условие честного сравнения. `random_state=42` делает эксперимент воспроизводимым.

> Для реального прогнозирования по времени нельзя случайно перемешивать последовательные измерения: там применяют разделение на прошлое и будущее. Здесь мы решаем задачу оценки температуры по параметрам текущего наблюдения.

In [ ]:
indices = np.arange(len(df))
train_idx, test_idx = train_test_split(indices, test_size=0.20, random_state=42)

X_simple_train = X_simple.iloc[train_idx]
X_simple_test = X_simple.iloc[test_idx]
X_multiple_train = X_multiple.iloc[train_idx]
X_multiple_test = X_multiple.iloc[test_idx]
y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print(f"Train: {len(train_idx)} наблюдений; Test: {len(test_idx)} наблюдений")
assert set(X_simple_test.index) == set(X_multiple_test.index)

## 5. Обучение простой линейной регрессии

Модель подбирает коэффициенты в уравнении `temperature ≈ b + w × current`.

In [ ]:
simple_model = LinearRegression()
simple_model.fit(X_simple_train, y_train)

pred_simple = simple_model.predict(X_simple_test)
print(f"Простая модель: temperature ≈ {simple_model.intercept_:.3f} + "
      f"({simple_model.coef_[0]:.3f}) × current")

## 6. Обучение множественной линейной регрессии

Теперь учитываем ток, напряжение и режим работы. Каждому признаку соответствует собственный коэффициент.

In [ ]:
multiple_model = LinearRegression()
multiple_model.fit(X_multiple_train, y_train)

pred_multiple = multiple_model.predict(X_multiple_test)

coef_table = pd.DataFrame({
    "Признак": X_multiple.columns,
    "Коэффициент": multiple_model.coef_
})
print(f"Свободный член b: {multiple_model.intercept_:.3f}")
display(coef_table.round(4))
print("Важно: коэффициенты режимов означают отличие от базовой категории, "
      "а влияние остальных признаков интерпретируется при прочих равных.")

## 7. MAE, MSE, RMSE и R² на одинаковой тестовой выборке

- **MAE** — средняя абсолютная ошибка в °C, меньше — лучше.
- **MSE** — средняя квадратичная ошибка в °C², меньше — лучше.
- **RMSE** — корень MSE в °C, больше штрафует крупные ошибки.
- **R²** — коэффициент детерминации; 1 — идеальное соответствие, 0 — качество базового прогноза средним, отрицательное значение — хуже этого базового прогноза.

In [ ]:
def evaluate_model(y_true, y_pred):
    """Возвращает метрики регрессии для одной выборки."""
    mse = mean_squared_error(y_true, y_pred)
    return {
        "MAE (°C)": mean_absolute_error(y_true, y_pred),
        "MSE (°C²)": mse,
        "RMSE (°C)": np.sqrt(mse),
        "R²": r2_score(y_true, y_pred)
    }

comparison = pd.DataFrame({
    "Простая (current)": evaluate_model(y_test, pred_simple),
    "Множественная (+ voltage, mode)": evaluate_model(y_test, pred_multiple)
}).T

display(comparison.round(3))
print("Проверка: показатели рассчитаны только на Test, не на Train.")

## 8. Наглядное сравнение ошибок

Диаграмма сравнивает MAE и RMSE на тех же тестовых примерах. Различие в величинах покажет, помогли ли дополнительные признаки.

In [ ]:
comparison[["MAE (°C)", "RMSE (°C)"]].plot(kind="bar", rot=0, figsize=(10, 5))
plt.title("Сравнение ошибок двух моделей на Test")
plt.ylabel("Ошибка, °C")
plt.xlabel("")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Реальная температура vs предсказанная

Пунктирная линия означает идеальный прогноз. Чем ближе к ней точки — тем точнее оценка.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
low = min(y_test.min(), pred_simple.min(), pred_multiple.min())
high = max(y_test.max(), pred_simple.max(), pred_multiple.max())

for ax, pred, title in zip(
    axes,
    [pred_simple, pred_multiple],
    ["Простая модель", "Множественная модель"]
):
    ax.scatter(y_test, pred, alpha=0.65)
    ax.plot([low, high], [low, high], "r--", label="Идеальный прогноз")
    ax.set(title=title, xlabel="Реальная температура, °C", ylabel="Прогноз, °C")
    ax.legend()
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 10. Сравнение остатков

Остаток = реальная температура − предсказанная. Хорошо, когда остатки группируются около нуля без явного рисунка. Особо большие остатки стоит разобрать отдельно.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, pred, title in zip(
    axes,
    [pred_simple, pred_multiple],
    ["Простая модель", "Множественная модель"]
):
    residuals = y_test.to_numpy() - pred
    ax.scatter(pred, residuals, alpha=0.65)
    ax.axhline(0, color="red", linestyle="--")
    ax.set(title=title, xlabel="Прогноз, °C", ylabel="Остаток, °C")
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 11. Прогноз температуры для новых параметров спутника

Предположим, мы получили новые параметры тока, напряжения и режима работы. **Обе модели получают одинаковый ток**, но множественная также использует напряжение и режим. Столбцы для категорий должны совпадать с теми, что применялись при обучении.

In [ ]:
new_data = pd.DataFrame({
    "current": [0.9, 1.4, 2.1, 2.4],
    "voltage": [27.8, 27.4, 27.0, 26.8],
    "mode": ["standby", "normal", "active", "active"]
})

# Простая модель использует только ток.
new_simple = new_data[["current"]]

# Применяем ту же схему кодирования, что и во время обучения.
new_multiple = pd.get_dummies(
    new_data, columns=["mode"], drop_first=False, dtype=int
).reindex(columns=X_multiple.columns, fill_value=0)

forecast = new_data.copy()
forecast["Простая модель, °C"] = simple_model.predict(new_simple)
forecast["Множественная модель, °C"] = multiple_model.predict(new_multiple)

display(forecast.round(2))
print("Это оценки модели, а не показания датчика температуры.")

## 12. Бонус: что будет, если добавить `power`?

Мощность равна произведению тока и напряжения (`power ≈ current × voltage`). Такой признак может частично дублировать информацию и осложнять интерпретацию коэффициентов. **Дополнительный признак не гарантирует улучшение на Test.**

In [ ]:
X_power = X_multiple.copy()
X_power["power"] = df["power"]

power_model = LinearRegression()
power_model.fit(X_power.iloc[train_idx], y_train)
pred_power = power_model.predict(X_power.iloc[test_idx])

comparison_extended = comparison.copy()
comparison_extended.loc["Множественная + power"] = evaluate_model(y_test, pred_power)
display(comparison_extended.round(3))

## 13. Выводы и вопросы студентам

1. У какой модели оказались меньше MAE и RMSE на тестовой выборке?
2. Как изменилось R² после добавления напряжения и режима работы?
3. Почему режим `mode` нельзя передать в `LinearRegression` просто как текст?
4. Что означают коэффициенты у категориальных признаков?
5. Всегда ли добавление `power` улучшает точность? Почему нет?
6. Почему сравнение на **одной** тестовой выборке важнее сравнения на разных случайных разбиениях?
7. Что произойдёт, если попробовать получить прогноз для тока, значительно превышающего значения в датасете?

**Главный вывод:** множественная регрессия может учитывать больше факторов, но пользу дополнительных признаков нужно проверять по метрикам на ранее не виденных данных.

**Следующая тема курса:** прогнозирование временных рядов — когда входом становятся предыдущие измерения, а целью является температура в будущем.